In [1]:
%pwd

'c:\\Users\\Amal\\Desktop\\End to End project Mlops mlflow github\\End-to-End-project-Mlops-Mlflow\\research'

In [2]:
%cd ..

c:\Users\Amal\Desktop\End to End project Mlops mlflow github\End-to-End-project-Mlops-Mlflow


C:\Users\Amal\AppData\Roaming\Python\Python39\site-packages\IPython\core\magics\osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [3]:
%pwd

'c:\\Users\\Amal\\Desktop\\End to End project Mlops mlflow github\\End-to-End-project-Mlops-Mlflow'

Update the entity

In [4]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir: Path
    train_data_path: Path
    test_data_path: Path   
    model_name: str
    alpha: float
    l1_ratio: float
    target_column: str

Update the configuration manager

In [5]:
from mlProject.constants import *
from mlProject.utils.common import read_yaml, create_directories

In [6]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])


    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config = self.config.model_trainer
        alpha = self.params.alpha
        l1_ratio = self.params.l1_ratio        
        schema = self.schema.TARGET_COLUMN

        create_directories([config.root_dir])

        model_trainer_config = ModelTrainerConfig(
            root_dir=config.root_dir,
            train_data_path = config.train_data_path,
            test_data_path = config.test_data_path,
            model_name = config.model_name,
            alpha = alpha,
            l1_ratio = l1_ratio,
            target_column = schema.name
        )

        return model_trainer_config

Update the components

In [7]:
import pandas as pd
import os
from mlProject import logger
from sklearn.linear_model import ElasticNet
import joblib

In [ ]:
class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config

    
    def train(self):
        
        train_data = pd.read_csv(self.config.train_data_path)
        test_data = pd.read_csv(self.config.test_data_path)


        train_x = train_data.drop([self.config.target_column], axis=1)
        test_x = test_data.drop([self.config.target_column], axis=1)
        train_y = train_data[[self.config.target_column]]
        test_y = test_data[[self.config.target_column]]


        lr = ElasticNet(alpha=self.config.alpha, l1_ratio=self.config.l1_ratio, random_state=42)
        lr.fit(train_x, train_y)
        
        
        # if the model name already exists, add 1 to the model name to have a new model name 
        model_name = self.config.model_name 
        base_name, extension = os.path.splitext(model_name)
        
        counter = 1
        new_name = model_name

        while os.path.exists(os.path.join(self.config.root_dir, new_name)):
            new_name = f"{base_name}{counter}{extension}"
            counter += 1

        joblib.dump(lr, os.path.join(self.config.root_dir, new_name))

Update the pipline

In [24]:
try:
    config = ConfigurationManager()
    model_trainer_config = config.get_model_trainer_config()
    model_trainer_config = ModelTrainer(config=model_trainer_config)
    model_trainer_config.train()
except Exception as e:
    raise e

[2026-06-04 16:53:24,169: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-06-04 16:53:24,174: INFO: common: yaml file: params.yaml loaded successfully]
[2026-06-04 16:53:24,183: INFO: common: yaml file: schema.yaml loaded successfully]


[2026-06-04 16:53:24,187: INFO: common: created directory at: artifacts]
[2026-06-04 16:53:24,192: INFO: common: created directory at: artifacts/model_trainer]
